# Producer 2

In [5]:
from kafka import KafkaProducer
import json
import random
import time
from datetime import datetime

In [6]:
# Подключаемся к Kafka
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",

    # Преобразуем Python-словарь в JSON
    value_serializer=lambda x: json.dumps(x).encode("utf-8"),

    # Ключ сообщения — ID клиента
    key_serializer=lambda x: str(x).encode("utf-8")
)

In [7]:
# Создаем словарик с ID продукта и ценами
products = {
    201: 1500,
    202: 2500,
    203: 5000,
    204: 7500
}

In [8]:
print("Producer started")
# Симуляция создания заказа в интернет-магазине
try:
    for i in range(10):
        # Генерируем случайный ID клиента 
        customer_id = random.randint(1, 5)
        # Выбираем случайный продукт из словаря
        product_id = random.choice(list(products.keys()))
        
        event = {
            "order_id": i + 1,
            "customer_id": customer_id,
            "product_id": product_id,
            "amount": products[product_id],
            "timestamp": datetime.now().isoformat()
        }

        # Отправляем событие в Kafka
        future = producer.send(
            "purchases",
            key=customer_id,
            value=event
        )

        # Ждем подтверждения от Kafka
        metadata = future.get(timeout=10)

        print(
            f"Sent: {event} | "
            f"partition={metadata.partition}, "
            f"offset={metadata.offset}"
        )

        time.sleep(3)

finally:
    producer.flush()
    producer.close()

Producer started
Sent: {'order_id': 1, 'customer_id': 1, 'product_id': 202, 'amount': 2500, 'timestamp': '2026-11-02T20:13:00.081252'} | partition=0, offset=60
Sent: {'order_id': 2, 'customer_id': 1, 'product_id': 203, 'amount': 5000, 'timestamp': '2026-11-02T20:13:03.246346'} | partition=0, offset=61
Sent: {'order_id': 3, 'customer_id': 4, 'product_id': 202, 'amount': 2500, 'timestamp': '2026-11-02T20:13:06.270894'} | partition=0, offset=62
Sent: {'order_id': 4, 'customer_id': 2, 'product_id': 204, 'amount': 7500, 'timestamp': '2026-11-02T20:13:09.284977'} | partition=0, offset=63
Sent: {'order_id': 5, 'customer_id': 5, 'product_id': 203, 'amount': 5000, 'timestamp': '2026-11-02T20:13:12.317609'} | partition=0, offset=64
Sent: {'order_id': 6, 'customer_id': 4, 'product_id': 203, 'amount': 5000, 'timestamp': '2026-11-02T20:13:15.332676'} | partition=0, offset=65
Sent: {'order_id': 7, 'customer_id': 3, 'product_id': 202, 'amount': 2500, 'timestamp': '2026-11-02T20:13:18.346451'} | parti